In [1]:
from futu import *
import pandas_ta as ta
import pandas as pd

import json
import talib

/opt/homebrew/lib/python3.10/site-packages/pandas_ta/__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)

In [3]:
quote_ctx = OpenQuoteContext(host='127.0.0.1', port=11111)

2025-07-02 10:32:15,276 | 58188 | [open_context_base.py] _send_init_connect_sync:311: InitConnect ok: conn_id=1, host=127.0.0.1, port=11111, user_id=11541642


In [193]:
# quote_ctx.close()

In [4]:
watchlists = quote_ctx.get_user_security_group(group_type = UserSecurityGroupType.ALL)

In [5]:
def get_watch_tickers(group_name="特别关注"):
	ret_code, heart_watch_list = quote_ctx.get_user_security(group_name=group_name)

	heart_watch_list = heart_watch_list[heart_watch_list['stock_type'] == 'STOCK']
	heart_watch_list = heart_watch_list[heart_watch_list['code'].str.startswith('HK')]

	watch_code_list = heart_watch_list['code'].tolist()

	return watch_code_list

In [6]:
trd_ctx = OpenSecTradeContext(filter_trdmarket=TrdMarket.HK, host='127.0.0.1', port=11111, security_firm=SecurityFirm.FUTUSECURITIES)

ret, position_df = trd_ctx.position_list_query()

position_tickers = position_df['code'].tolist()

2025-07-02 10:32:22,781 | 58188 | [open_context_base.py] _send_init_connect_sync:311: InitConnect ok: conn_id=2, host=127.0.0.1, port=11111, user_id=11541642


In [7]:
display(position_df)

,code,stock_name,position_market,qty,can_sell_qty,cost_price,cost_price_valid,average_cost,diluted_cost,market_val,...,today_buy_qty,today_buy_val,today_pl_val,today_trd_val,today_sell_qty,today_sell_val,position_side,unrealized_pl,realized_pl,currency
0,HK.03690,美团-W,HK,500.0,500.0,132.500,True,132.500,132.500,62300.0,...,0.0,0.0,-350.0,0.0,0.0,0.0,LONG,-3950.0000,0.0000,HKD
1,HK.01788,国泰君安国际,HK,10000.0,10000.0,4.730,True,4.730,4.730,34600.0,...,0.0,0.0,300.0,0.0,0.0,0.0,LONG,-12700.0000,0.0000,HKD
2,HK.01398,工商银行,HK,5000.0,5000.0,6.360,True,6.360,6.360,31650.0,...,5000.0,31800.0,-150.0,31800.0,0.0,0.0,LONG,-150.0000,0.0000,HKD
3,HK.01211,比亚迪股份,HK,1000.0,1000.0,265.600,True,265.600,265.600,121600.0,...,0.0,0.0,-900.0,0.0,0.0,0.0,LONG,-144000.0000,0.0000,HKD
4,HK.00700,腾讯控股,HK,283.0,283.0,62.259,True,267.338,62.259,141783.0,...,0.0,0.0,-566.0,0.0,0.0,0.0,LONG,66126.2635,556180.7365,HKD


In [8]:
ret, hengsheng_df = quote_ctx.get_plate_stock('HK.HSI Constituent Stocks')  # 恒指成分

hengsheng_tickers = hengsheng_df['code'].tolist()

ret, hot_df = quote_ctx.get_plate_stock('HK.LIST1600')
hot_tickers = hot_df['code'].tolist()

In [9]:

# 腾讯，美团，比亚迪，中芯国际
# for ticker_id in ['HK.00700', 'HK.03690', 'HK.01211', 'HK.00981']:
#    message[ticker_id] = process_super_trend(ticker_id=ticker_id)

config_tickers_list = []
config_tickers_list = get_watch_tickers(group_name="特别关注")
config_tickers_list += get_watch_tickers(group_name="港股")
config_tickers_list += get_watch_tickers(group_name="沪深")

config_tickers_list += hengsheng_tickers
config_tickers_list += hot_tickers
config_tickers_list += position_tickers

config_tickers_list = list(set(config_tickers_list))  # 去重

config_tickers = pd.DataFrame({'ticker_id': config_tickers_list})

In [10]:
config_tickers.shape

(509, 1)

In [ ]:
if False:
	_, test_df, _ = quote_ctx.request_history_kline(
		"HK.00700", start='2000-01-01', end='2025-12-31', max_count=10000, session=Session.ALL)
	display(test_df.sort_values(by='time_key', ascending=False).head(10))

,code,name,time_key,open,close,high,low,pe_ratio,turnover_rate,volume,turnover,change_rate,last_close
2462,HK.00700,腾讯控股,2025-07-02 00:00:00,508.0,500.5,508.5,500.0,21.889,0.00088,8020968,4.040888e+09,-0.497018,503.0
2461,HK.00700,腾讯控股,2025-06-30 00:00:00,509.0,503.0,511.5,503.0,21.998,0.00201,18464531,9.333594e+09,-1.949318,513.0
2460,HK.00700,腾讯控股,2025-06-27 00:00:00,512.0,513.0,514.5,510.0,22.461,0.00165,15181667,7.767267e+09,0.000000,513.0
2459,HK.00700,腾讯控股,2025-06-26 00:00:00,512.0,513.0,513.0,507.5,22.461,0.00163,15000643,7.665271e+09,0.097561,512.5
2458,HK.00700,腾讯控股,2025-06-25 00:00:00,512.0,512.5,515.0,508.0,22.439,0.00192,17592461,8.995908e+09,0.588813,509.5
2457,HK.00700,腾讯控股,2025-06-24 00:00:00,508.0,509.5,511.0,504.0,22.308,0.00194,17768858,9.026969e+09,1.091270,504.0
2456,HK.00700,腾讯控股,2025-06-23 00:00:00,498.2,504.0,504.0,494.6,22.067,0.00155,14252887,7.125786e+09,-0.296736,505.5
2455,HK.00700,腾讯控股,2025-06-20 00:00:00,504.5,505.5,505.5,496.0,22.132,0.00225,20622275,1.036414e+10,1.506024,498.0
2454,HK.00700,腾讯控股,2025-06-19 00:00:00,503.0,498.0,506.0,496.0,21.803,0.00211,19387767,9.674346e+09,-1.968504,508.0
2453,HK.00700,腾讯控股,2025-06-18 00:00:00,510.0,508.0,511.0,503.5,22.241,0.00166,15187515,7.682823e+09,-1.071081,513.5


In [30]:
def get_ticker_history(ticker_id):
	output_file = f"./data/{ticker_id}.csv"
	max_date = None
	curr_date = datetime.today().date().strftime('%Y-%m-%d')
	last_date = (datetime.today() - pd.Timedelta(days=1)).date().strftime('%Y-%m-%d')
	curr_hour = datetime.now().hour
	if os.path.isfile(output_file):
		data = pd.read_csv(output_file)
		if not data.empty:
			max_date = data['time_key'].max()
			max_date = pd.to_datetime(max_date).strftime('%Y-%m-%d')
			if (curr_hour > 18 and max_date >= curr_date) or (curr_hour <= 18 and max_date >= last_date):  
				# 18点以后，尝试获取今天的数据,18点之前，尝试获取昨天的数据
				return data

	ret, data, page_req_key = quote_ctx.request_history_kline(
		ticker_id, start='2000-01-01', end='2025-12-31', max_count=10000, session=Session.ALL)
	time.sleep(0.51)   # qps limit 2
	if ret != 0:
		print(f'Error fetching history for {ticker_id}: {data}')
		return None
	
	data.to_csv(output_file, index=False)
	return data

In [31]:
ticker_history = pd.DataFrame({
    key: [] for key in
    ['code', 'name', 'time_key', 'open', 'close', 'high', 'low', 'pe_ratio',
       'turnover_rate', 'volume', 'turnover', 'change_rate', 'last_close']
})

for index, ticker_id in enumerate(config_tickers_list):
    curr_ticker_history = get_ticker_history(ticker_id)
    if curr_ticker_history is not None and not curr_ticker_history.empty:
        ticker_history = pd.concat([ticker_history, curr_ticker_history], axis=0)
        print(f"Processed {ticker_id} {index + 1}/{len(config_tickers_list)}, {curr_ticker_history.shape}, {curr_ticker_history['time_key'].max()}")

Processed HK.01985 1/509, (2044, 13), 2025-06-30 00:00:00
Processed HK.01888 2/509, (2462, 13), 2025-07-02 00:00:00
Processed HK.00101 3/509, (2463, 13), 2025-07-02 00:00:00
Processed HK.00570 4/509, (2451, 13), 2025-07-02 00:00:00
Processed HK.06969 5/509, (1223, 13), 2025-07-02 00:00:00
Processed HK.06086 6/509, (241, 13), 2025-07-02 00:00:00
Processed HK.02587 7/509, (123, 13), 2025-07-02 00:00:00
Processed HK.08162 8/509, (2204, 13), 2025-07-02 00:00:00
Processed HK.01911 9/509, (1310, 13), 2025-07-02 00:00:00
Processed HK.02228 10/509, (258, 13), 2025-07-02 00:00:00
Processed HK.02050 11/509, (7, 13), 2025-07-02 00:00:00
Processed HK.00279 12/509, (2025, 13), 2025-07-02 00:00:00
Processed HK.01093 13/509, (2462, 13), 2025-07-02 00:00:00
Processed HK.01428 14/509, (2461, 13), 2025-07-02 00:00:00
Processed HK.01497 15/509, (378, 13), 2025-06-30 00:00:00
Processed HK.02495 16/509, (240, 13), 2025-07-02 00:00:00
Processed HK.03330 17/509, (2463, 13), 2025-07-02 00:00:00
Processed HK.0

In [1]:
# ticker_history[ticker_history['code']=='HK.00700'].shape
ticker_history['code'].drop_duplicates().shape

quote_ctx.close()

NameError: name 'ticker_history' is not defined

In [47]:
def super_trend_daily(row, ticker_history, test=None):
	ticker_id = row['ticker_id']
	data = ticker_history[ticker_history['code'] == ticker_id].reset_index(drop=True)

	row['close'] = None
	row['ticker_name'] = None
	row['supert'] = None
	row['supertl'] = None
	row['supertd'] = None
	row['supertd_prev'] = None

	row['supertd_change'] = None
	row['time_key'] = None

	if data is None or data.empty:
		print(f"Failed to fetch data for {ticker_id}")
		return row


	ticker_name = data.iloc[0]['name'].strip()
	row['ticker_name'] = ticker_name

	SUPERTd = ta.supertrend(high=data['high'], low=data['low'], close=data['close'], length=10, multiplier=3.0, fillna=0.0)

	if SUPERTd is None or SUPERTd.empty:
		print(f"Failed to calculate Supertrend for {ticker_id} | 可能是上市时间太短。 {data.shape}")
		return row
	
	data['supert'] = SUPERTd['SUPERT_10_3.0']
	data['supertl'] = SUPERTd['SUPERTl_10_3.0']
	data['supertd'] = SUPERTd['SUPERTd_10_3.0']

	data['supertd_prev'] = data['supertd'].shift(1)

	data = data.assign(
		supertd_change = 0
	)
	condition = (data['supertd'] > 0) & (data['supertd_prev'] < 0)
	data.loc[condition, 'supertd_change'] = 1

	condition = (data['supertd'] < 0) & (data['supertd_prev'] > 0)
	data.loc[condition, 'supertd_change'] = -1

	last_row = data.iloc[-1]

	message  = ""

	if last_row['supertd_change'] > 0 and last_row['supertd'] > 0 and last_row['close'] > last_row['supertl']:
		message = f"{ticker_id} 向上，买入机会"
	elif last_row['supertd_change'] < 0 and last_row['supertd'] < 0 and last_row['close'] < last_row['supertl']:
		message = f"{ticker_id} 向下，卖出机会"
	elif last_row['supertd'] > 0:
		message = f"{ticker_id} 向上，但需谨慎"
	elif last_row['supertd'] < 0:
		message = f"{ticker_id} 向下，但需谨慎"
	else:
		message = f"{ticker_id} 无明显趋势"
	row['close'] = last_row['close']

	row['supert'] = last_row['supert']
	row['supertl'] = last_row['supertl']
	row['supertd'] = last_row['supertd']
	row['supertd_prev'] = last_row['supertd_prev']
	row['supertd_change'] = last_row['supertd_change']
	row['time_key'] = last_row['time_key']
	return row

In [56]:
def cal_technicals(row, ticker_data_df, test=None):
	ticker_data = ticker_data_df[ticker_data_df['code'] == row['ticker_id']].reset_index(drop=True)

	row['EMA_144_daily'] = None
	row['EMA_169_daily'] = None
	row['SMA_5_Cross_SMA_20'] = 0

	if ticker_data.empty:
		return row
	
	indicators = [
		(talib.EMA, "EMA_13_daily", 13),
		(talib.EMA, "EMA_21_daily", 20),
		(talib.EMA, "EMA_144_daily", 144),
		(talib.EMA, "EMA_169_daily", 169),
		(talib.SMA, "SMA_5_daily", 5),
		(talib.SMA, "SMA_10_daily", 10),
		(talib.SMA, "SMA_20_daily", 20),
		(talib.SMA, "SMA_50_daily", 50),
		(talib.SMA, "SMA_100_daily", 100),
	]

	for func, name, period in indicators:
		ticker_data[name] = func(ticker_data['close'], timeperiod=period)


	ticker_data = ticker_data.assign(SMA_5_Cross_SMA_20=0,)
	ticker_data['SMA_5_daily_prev'] = ticker_data['SMA_5_daily'].shift(1)
	ticker_data['SMA_20_daily_prev'] = ticker_data['SMA_20_daily'].shift(1)

	condition = (ticker_data['SMA_5_daily'] > ticker_data['SMA_20_daily']) & (ticker_data['SMA_5_daily_prev'] < ticker_data['SMA_20_daily_prev'])
	ticker_data.loc[condition, 'SMA_5_Cross_SMA_20'] = 1
	condition = (ticker_data['SMA_5_daily'] < ticker_data['SMA_20_daily']) & (ticker_data['SMA_5_daily_prev'] > ticker_data['SMA_20_daily_prev'])
	ticker_data.loc[condition, 'SMA_5_Cross_SMA_20'] = -1

	ticker_data = ticker_data.drop(columns=['SMA_5_daily_prev', 'SMA_20_daily_prev'])

	last_row = ticker_data.iloc[-1]

	row['EMA_144_daily'] = last_row['EMA_144_daily']
	row['EMA_169_daily'] = last_row['EMA_169_daily']
	row['SMA_5_Cross_SMA_20'] = last_row['SMA_5_Cross_SMA_20']
	row['vegas'] = 0
	if row['close'] > row['EMA_144_daily'] and row['close'] > row['EMA_169_daily']:
		row['vegas'] = 1
	elif row['close'] < row['EMA_144_daily'] and row['close'] < row['EMA_169_daily']:
		row['vegas'] = -1
	return row


In [52]:
df = config_tickers.apply(super_trend_daily, axis=1, args=(ticker_history, None))

print('cal super trend success.')

Failed to calculate Supertrend for HK.02050 | 可能是上市时间太短。 (7, 13)
Failed to calculate Supertrend for HK.02508 | 可能是上市时间太短。 (4, 13)
Failed to calculate Supertrend for HK.06883 | 可能是上市时间太短。 (4, 13)
Failed to fetch data for HK.06613
Failed to calculate Supertrend for HK.03880 | 可能是上市时间太短。 (2, 13)
Failed to calculate Supertrend for HK.09678 | 可能是上市时间太短。 (2, 13)
Failed to calculate Supertrend for HK.02619 | 可能是上市时间太短。 (5, 13)
Failed to fetch data for HK.02592
Failed to calculate Supertrend for HK.02643 | 可能是上市时间太短。 (5, 13)
Failed to calculate Supertrend for HK.06168 | 可能是上市时间太短。 (4, 13)
Failed to calculate Supertrend for HK.03288 | 可能是上市时间太短。 (9, 13)
Failed to calculate Supertrend for HK.02609 | 可能是上市时间太短。 (7, 13)
Failed to calculate Supertrend for HK.06603 | 可能是上市时间太短。 (2, 13)
Failed to fetch data for HK.02915
Failed to calculate Supertrend for HK.02617 | 可能是上市时间太短。 (7, 13)


In [57]:
df = df.apply(cal_technicals, axis=1, args=(ticker_history, None))
print('cal technicals success.')

cal technicals success.


In [61]:
SMA520_df = df[df['SMA_5_Cross_SMA_20'] > 0].reset_index(drop=True)

In [37]:
bull_df = df[(df['supertd'] > 0) & (df['supertd_change'] > 0) & (df['vegas'] > 0)]
bear_df = df[(df['supertd'] < 0) & (df['supertd_change'] < 0) & (df['vegas'] < 0)]

new_ticker_df = df[df['ticker_name'].isnull()]

df = df.dropna(subset=['ticker_name'])   # ticker name为打新股未上市

In [38]:
rev_df = position_df.merge(df, left_on='code', right_on='ticker_id', how='left')
rev_df['prof_rate'] = rev_df['close'] / rev_df['average_cost'] - 1

rev_df = rev_df[['code', 'time_key', 'stock_name', 'average_cost',
		'position_side', 'close', 'prof_rate',
		'market_val', 'supertd', 'supertd_prev', 'supertd_change', 'vegas']]

In [39]:
display(rev_df)

,code,time_key,stock_name,average_cost,position_side,close,prof_rate,market_val,supertd,supertd_prev,supertd_change,vegas
0,HK.03690,2025-07-02 00:00:00,美团-W,132.500,LONG,124.20,-0.062642,62300.0,-1.0,-1.0,0.0,-1.0
1,HK.01788,2025-07-02 00:00:00,国泰君安国际,4.730,LONG,3.47,-0.266385,34600.0,-1.0,-1.0,0.0,1.0
2,HK.01398,2025-07-02 00:00:00,工商银行,6.360,LONG,6.36,0.000000,31650.0,1.0,1.0,0.0,1.0
3,HK.01211,2025-07-02 00:00:00,比亚迪股份,265.600,LONG,122.00,-0.540663,121600.0,-1.0,-1.0,0.0,1.0
4,HK.00700,2025-07-02 00:00:00,腾讯控股,267.338,LONG,502.00,0.877773,141783.0,1.0,1.0,0.0,1.0


In [40]:
display(new_ticker_df)

,EMA_144_daily,EMA_169_daily,close,supert,supertd,supertd_change,supertd_prev,supertl,ticker_id,ticker_name,time_key,vegas
106,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HK.06613,None,None,NaN
162,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HK.02592,None,None,NaN
487,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HK.02915,None,None,NaN


In [ ]:
display(bull_df)

,EMA_144_daily,EMA_169_daily,close,supert,supertd,supertd_change,supertd_prev,supertl,ticker_id,ticker_name,time_key,vegas
308,7.540169,7.574390,7.760,7.019796,1.0,1.0,-1.0,7.019796,HK.00868,信义玻璃,2025-07-02 00:00:00,1.0
312,0.078146,0.077746,0.088,0.065619,1.0,1.0,-1.0,0.065619,HK.01647,雄岸科技,2025-07-02 00:00:00,1.0


In [ ]:
display(bear_df)

,EMA_144_daily,EMA_169_daily,close,supert,supertd,supertd_change,supertd_prev,supertl,ticker_id,ticker_name,time_key,vegas
356,73.799255,72.701713,65.000,86.257425,-1.0,-1.0,1.0,0.0,HK.02479,天聚地合,2025-03-31 00:00:00,-1.0
384,0.042531,0.042541,0.037,0.049121,-1.0,-1.0,1.0,0.0,HK.00491,英皇文化产业,2025-07-02 00:00:00,-1.0


In [62]:
display(SMA520_df)

,EMA_144_daily,EMA_169_daily,close,supert,supertd,supertd_change,supertd_prev,supertl,ticker_id,ticker_name,time_key,vegas,SMA_5_Cross_SMA_20
0,0.031930,0.033281,0.024,0.024653,-1.0,0.0,-1.0,0.000000,HK.08532,宝发控股,2025-06-30 00:00:00,-1.0,1
1,0.040342,0.040847,0.048,0.040367,1.0,0.0,1.0,0.040367,HK.01132,橙天嘉禾,2025-07-02 00:00:00,1.0,1
2,0.617460,0.714139,0.380,0.335001,1.0,0.0,1.0,0.335001,HK.02438,出门问问,2025-07-02 00:00:00,-1.0,1
3,5.483593,5.539478,5.000,5.123113,-1.0,0.0,-1.0,0.000000,HK.02202,万科企业,2025-07-02 00:00:00,-1.0,1
4,37.131785,37.264927,35.500,37.463895,-1.0,0.0,-1.0,0.000000,HK.09896,名创优品,2025-07-02 00:00:00,-1.0,1
5,0.453687,0.434801,0.650,0.554588,1.0,0.0,1.0,0.554588,HK.00169,万达酒店发展,2025-07-02 00:00:00,1.0,1
6,26.805130,27.349706,21.150,22.022747,-1.0,0.0,-1.0,0.000000,HK.02410,同源康医药-B,2025-07-02 00:00:00,-1.0,1
7,1.306030,1.364989,1.410,1.140680,1.0,0.0,1.0,1.140680,HK.02150,奈雪的茶,2025-07-02 00:00:00,1.0,1
8,10.229048,10.512169,9.500,8.055602,1.0,0.0,1.0,8.055602,HK.06865,福莱特玻璃,2025-07-02 00:00:00,-1.0,1
9,3.312532,3.250893,3.220,3.443783,-1.0,0.0,-1.0,0.000000,HK.03738,阜博集团,2025-07-02 00:00:00,-1.0,1


In [46]:
df[(df['vegas'] > 0) & (df['supertd'] > 0) & (df['close'] > df['supertl'])].reset_index(drop=True)

,EMA_144_daily,EMA_169_daily,close,supert,supertd,supertd_change,supertd_prev,supertl,ticker_id,ticker_name,time_key,vegas
0,8.012562,7.877448,9.780,8.838706,1.0,0.0,1.0,8.838706,HK.01888,建滔积层板,2025-07-02 00:00:00,1.0
1,6.290089,6.301469,7.590,6.970516,1.0,0.0,1.0,6.970516,HK.00101,恒隆地产,2025-07-02 00:00:00,1.0
2,3.749877,3.904095,4.730,3.813818,1.0,0.0,1.0,3.813818,HK.01911,华兴资本控股,2025-07-02 00:00:00,1.0
3,0.035079,0.035257,0.045,0.033389,1.0,0.0,1.0,0.033389,HK.00279,裕承科金,2025-07-02 00:00:00,1.0
4,6.095045,5.993607,7.940,7.203661,1.0,0.0,1.0,7.203661,HK.01093,石药集团,2025-07-02 00:00:00,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
255,0.074226,0.073722,0.093,0.080554,1.0,0.0,1.0,0.080554,HK.01621,域高国际控股,2025-07-02 00:00:00,1.0
256,5.590627,5.537246,6.050,5.738168,1.0,0.0,1.0,5.738168,HK.02386,中石化炼化工程,2025-07-02 00:00:00,1.0
257,469.042063,461.162714,502.000,487.569245,1.0,0.0,1.0,487.569245,HK.00700,腾讯控股,2025-07-02 00:00:00,1.0
258,27.384294,27.326975,29.550,26.589180,1.0,0.0,1.0,26.589180,HK.00772,阅文集团,2025-07-02 00:00:00,1.0
